In [1]:
import pandas as pd
import sqlite3

# Load the CSV
df = pd.read_csv('data/bank_churn.csv')

# Basic checks
print(df.shape)
print(df.head())
print(df.isnull().sum())

(4999, 12)
   customer_id  credit_score country  gender  age  tenure    balance  \
0     15634602           619  France  Female   42       2       0.00   
1     15647311           608   Spain  Female   41       1   83807.86   
2     15619304           502  France  Female   42       8  159660.80   
3     15701354           699  France  Female   39       1       0.00   
4     15737888           850   Spain  Female   43       2  125510.82   

   products_number  credit_card  active_member  estimated_salary  churn  
0                1            1              1         101348.88      1  
1                1            0              1         112542.58      0  
2                3            1              0         113931.57      1  
3                2            0              0          93826.63      0  
4                1            1              1          79084.10      0  
customer_id         0
credit_score        0
country             0
gender              0
age                 0
te

In [2]:
# Push to SQLite
conn = sqlite3.connect('churn.db')
df.to_sql('customers', conn, if_exists='replace', index=False)
print("Data loaded into SQLite successfully")

Data loaded into SQLite successfully


In [3]:
query = """
SELECT products_number,
       COUNT(*) as total_customers,
       SUM(churn) as churned_customers,
       ROUND(SUM(churn) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY products_number
ORDER BY products_number
"""

result = pd.read_sql_query(query, conn)
print(result)

   products_number  total_customers  churned_customers  churn_rate
0                1             2576                733       28.45
1                2             2262                172        7.60
2                3              130                110       84.62
3                4               31                 31      100.00


Day 1

This query looks at metrics related to churn for customers with a certain number of products.

Here we understand that customers with:
 - 1 products - are not committed enough, some needs not being meet, easy to leave
  - 2 products - sweet spot, they are committed and satisfied
    - 3-4 products - highest engagements, yet we see the highest churn

the goal of the bank should be to identify unmeet needs for customers with 1 product and aim to get them to 2. Also for customers with 3-4 products, to actively manage their experience before they leave, as high engagements comes with high expectations.

In [ ]:
# Day 2 - 8am: analyze churn by country and gender, write one insight per query

In [4]:
query = """
select country,
count() as total_customers,
sum(churn) as churned_customers, round(sum(churn) * 100.0 / count(), 2) as churn_rate
from customers
group by country
 """

result = pd.read_sql_query(query, conn)
print(result)

   country  total_customers  churned_customers  churn_rate
0   France             2506                437       17.44
1  Germany             1255                394       31.39
2    Spain             1238                215       17.37


Germany(31.39) has nearly double the churn rate of both France(17.44) and Spain(17.37). Bank needs to understand the cultural differences in banking behavior, competitive landscape for the german market before adopting retention strategies.

In [5]:
query = """
    select gender,
       count() as total_customers,
       sum(churn) as churned_customers,
       round(sum(churn) * 100.0 / count(), 2) as churn_rate
    from customers
    group by gender
"""

result = pd.read_sql_query(query, conn)
print(result)

   gender  total_customers  churned_customers  churn_rate
0  Female             2316                597       25.78
1    Male             2683                449       16.73


With the query result showing female having a higher churn rate than men, we hypothize that salary differences may contribute to the gap, as high earners may hold more products.

In [6]:
query = """
select min(age), max(age) from customers;
"""

result = pd.read_sql_query(query, conn)
print(result)

   min(age)  max(age)
0        18        88


In [7]:
# age group

query = """
select
    case
        when age between 18 and 28 then '20s'
        when age between 29 and 38 then '30s'
        when age between 39 and 48 then '40s'
        when age between 49 and 58 then '50s'
        when age between 59 and 68 then '60s'
        when age between 69 and 78 then '70s'
        when age between 79 and 88 then '80s'
    end as AgeGroup,
count(*) as total_customers,
sum(churn) as churned_customers,
round(sum(churn) * 100.0 / count(), 2) as churn_rate
from customers
group by AgeGroup;
"""

result = pd.read_sql_query(query, conn)
print(result)

  AgeGroup  total_customers  churned_customers  churn_rate
0      20s              646                 50        7.74
1      30s             2122                218       10.27
2      40s             1449                417       28.78
3      50s              479                267       55.74
4      60s              217                 85       39.17
5      70s               75                  8       10.67
6      80s               11                  1        9.09


Age Group Insight

This query splits the ages in to groups with 10 year gaps betweeen them. People in their 50s(55.74%) has the highest churn rate, we can hypothize that this period is when seasoned professsional near the end of their career, they would have needs such as retirement planning, wealth management. if they realize those needs are not being met they start shopping around. for the 40s, people in their group are at their peak earning period are likely to have needs such as mortgage, investments, savings for kids. also we see our lowest churn rate at 20s and 80s, these are people ideally with the lowest earning as they are either starting out or winding down, so they wouldn't need as much financial needs as other age groups.

Week 1 - Summary

query 1
- we looked at the churn rate of customers that own a certain number of products
- there is agoal to get customers with 1 products to 2 as that a sweet spot and less likely to churn with 2 products.
- we identified that customer with 3-4 products, has an enormous churn rate indiviuals - indicating that there something wrong with products or their expectations are not being meet as with the volume of products, it should signal high engagement

query 2
- germany has nearly double the churn rate of both france and spain
- we need to understand the banking culture within germany and tailor a retention strategy that meets their needs
- france and spain bothe a churn rate of 17% with spain actually be the lowest

query 3
- we found that women have a higher churn rate than me
- it hypothesized that the gap in salary differences has play a role their need for more products compared to men.
- the bank has more male customers than female.

query 4
- we identify that people in their 50s had the highest churn rate 55.74%.
- The 20s age group had the lowest churn rate, followed by people in their 80s. We hypothesized that at both stages earners are low - menaing less needs to be attended to.
- the 40s age appealed to me as thats an area with the highest earner potential and needs such as mortgages, saving for kids college are meet.

# Week 1 Summary

We identified that with more products comes higher expections. Retention strageies need to be tailored to customers with 3-4 products as they are highly engaged in the bank.

Germany has the highest churn rate amongst the countries we are present in. We need to revise a retention strategy that understands the banking of culture of Germany and meets their needs.

people in their 50s had the highest churn rate amongst other age. This is a cause of concern as these individuals are nearing retirements and ideally would shopping around for products that aid retirement plans and wealth managements.

# Week 2 - proving  original hypothesis — that high-balance, low-product customers represent the highest churn risk.


In [9]:
#balance tier
query = """
select MIN(balance) as min_balance,
    MAX(balance) as max_balance,
    ROUND(AVG(balance), 2) as avg_balance from customers;
"""

result = pd.read_sql_query(query, conn)
print(result)

   min_balance  max_balance  avg_balance
0          0.0    250898.09     77067.35


In [22]:
#churn by balance tier
query = """
select
    case
        when balance <= 60000 then 'low tier'
        when balance between 60000.01 and 90000 then 'middle tier'
        when balance between 90000.01 and 150000 then 'upper tier'
        when balance between 150000.01 and 250898.09 then 'high tier'
else 'uncategorized'
end as BalanceTiers,
count(*) as total_customers,
sum(churn) as churned_customers,
round(sum(churn) * 100.0 / count(), 2) as churn_rate
from customers
group by BalanceTiers;
"""

result = pd.read_sql_query(query, conn)
print(result)

  BalanceTiers  total_customers  churned_customers  churn_rate
0    high tier              497                113       22.74
1     low tier             1869                299       16.00
2  middle tier              426                 74       17.37
3   upper tier             2207                560       25.37


Our result shows that mid to high balance customers are most at risk to churn. We see the lowest churn rate at our low tier customers. It contradicts the hypothesis that the wealthiest are more highly to churn. The results shows them at 22.74% churn rate, with that balance it understandable that they'll have more products associated to them, and they'll have more to lose switching.

the bank needs to reach out to customer in the upper tier groups to understand what stages they are in their lives, ask them about their satisfaction with current services and know if there's anything that can be done for them.

In [23]:
#credit score
query = """
SELECT
    MIN(credit_score) as min_score,
    MAX(credit_score) as max_score,
    ROUND(AVG(credit_score), 2) as avg_score
FROM customers
"""

result = pd.read_sql_query(query, conn)
print(result)

   min_score  max_score  avg_score
0        350        850     650.26


In [29]:
query = """
select
    case
        when credit_score between 300 and 579 then 'poor'
        when credit_score between 580 and 669 then 'fair'
        when credit_score between 670 and 739 then 'good'
        when credit_score between 740 and 799 then 'very good'
        when credit_score between 800 and 850 then 'excellent'
end as CreditScore,
count(*) as total_customers,
sum(churn) as churned_customers,
round(sum(churn) * 100.0 / count(), 2) as churn_rate
from customers
group by CreditScore;
"""

result = pd.read_sql_query(query, conn)
print(result)

  CreditScore  total_customers  churned_customers  churn_rate
0   excellent              329                 66       20.06
1        fair             1655                341       20.60
2        good             1196                248       20.74
3        poor             1201                267       22.23
4   very good              618                124       20.06


The query result shows a close cluster of churn rate between 20.06 - 22.23 for all credit score groups. With how close they are it hard to say that its a strong predictor of churn in this dataset, with one exception to the poor credit customers. With every other group being approx 20, the poor credit customer churn is slightly elevated(22.23) with a clear business explanation backed by industry standards stating that customers in this tier are high risk. A reccomendation to the bank is to reevaluate the concurrent needs for these customers, and gradual limit services to what the customer can maintain.

Week 2 Summary

- We discovered that credit score isn't a strong predictor of churn in this dataset due to the closeness in range of churn rate(20.06 - 22.23), with exception to poor credit customer as they are the only one that sits outside the 20% approximation with a 22.23% churn rate.
- We discovered that our initial hypothesis stating that our highest earners are most likely to churn was wrong. We found out upper tier(25.37%) customers are the most likely to churn also low tier(16.00%) customer had the lowest.

- we came up with two recommendations this week, based on credit score we recommend a reevaluation done towards the needs of customers with poor credit score to understand what product meets their pressing need. With the upper balance, It's recommended to reach to understand what stages they are in their lives to better tailor their needs.

# Week 3 - 10am: identify 3 strongest patterns across all queries, build core narrative

Country

With our presence across the countries - France(17.44), Germany(31.39), Spain(17.37). Germany appeared to have the highest churn rate with is about double the churn rate of France and Spain. This matters to us because that indicates something wrong in Germany as a whole. The bank needs to understand the cultural differences, market landscape and tailor products and retention strategies towards the banking culture of the specific country.

Age Group

We looked at the churn rate across 7 age groups - 20s(7.74), 30s(10.27), 40s(28.78), 50s(55.74), 60s(39.17), 70s(10.67), 80s(9.09). We see our highest churn for people in their 50s, with the lowest being customers in their 20s and 80s. Age is important to us as it indirectly tells us what stages of life our customers are currently at. Ideally people in their 50s are nearing retirements and with that comes needs to for retirement planning and wealth management. We need to make them aware that we have products that'll meet those needs before they start seeking them elsewhere.

Balance Tiers

Here we created buckets for balances as tiers and looked at their churn rate - low tier(16.00), middle tier(17.37), upper tier(25.37), high tier(22.74). Customers in the upper tier had the highest churn rate which contradicts our initial hypothesis that the highest earner are the most likely to churn. This is an area where we need to check what stage they are in their lives, although not the highest earners they do have significant investment in our products. We need to evaluate their needs and make sure they are satisfied with what we offer.

Key Findings and Recommendations

In our exploration we found various factors related to churn rate, the ones we determined that had the large impact were Country, Age and Balance.
For Age, we looked at the churn rate across 7 age groups - 20s, 30s, 40s, 50s, 60s, 70s, 80s. We see our highest churn for people in their 50s, with the lowest being customers in their 20s and 80s. Age is important to us as it indirectly tells us what stages of life our customers are currently at. Ideally people in their 50s are nearing retirement and with that comes needs to for retirement planning and wealth management. We need to make them aware that we have products that'll meet those needs before they start seeking them elsewhere.

For Country, we have presence across three countries -France, Germany, Spain so we looked at their churn rate and Germany appeared to have the highest churn rate (31.39)which is about double the churn rate of France(17.44) and Spain(17.37). This matters to us because that indicates something wrong in Germany as a whole. The bank needs to understand the cultural differences, market landscape and tailor products and retention strategies towards the banking culture of the specific country.

For Balance Tiers, here we created buckets for balances as tiers and looked at their churn rate - low tier(16.00), middle tier(17.37), upper tier(25.37), high tier(22.74). Customers in the upper tier had the highest churn rate which contradicts our initial hypothesis that the highest earner are the most likely to churn. This is an area where we need to check what stage they are in their lives, although not the highest earners they do have significant investment in our products. We need to evaluate their needs and make sure they are satisfied with what we offer.


In [4]:
# Export query results for Power BI

#churn by product number
query1 = """
SELECT products_number,
       COUNT(*) as total_customers,
       SUM(churn) as churned_customers,
       ROUND(SUM(churn) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY products_number
"""

#churn by country
query2 = """
SELECT country,
       COUNT(*) as total_customers,
       SUM(churn) as churned_customers,
       ROUND(SUM(churn) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY country
"""

#churn by gender
query3 = """
SELECT gender,
       COUNT(*) as total_customers,
       SUM(churn) as churned_customers,
       ROUND(SUM(churn) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY gender
"""

#churn by age
query4 = """
SELECT
    CASE
        when age <= 28 then '20s'
        when age between 29 and 38 then '30s'
        when age between 39 and 48 then '40s'
        when age between 49 and 58 then '50s'
        when age between 59 and 68 then '60s'
        when age between 69 and 78 then '70s'
        else '80s'
    end as age_group,
    COUNT(*) as total_customers,
    SUM(churn) as churned_customers,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY age_group
"""

#churn by balance
"""
query5 = """
SELECT
    CASE
        when balance <= 60000 then 'low tier'
        when balance between 60000.01 and 90000 then 'middle tier'
        when balance between 90000.01 and 150000 then 'upper tier'
        else 'high tier'
    end as balance_tier,
    COUNT(*) as total_customers,
    SUM(churn) as churned_customers,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY balance_tier
"""
"""

pd.read_sql_query(query1, conn).to_csv('data/churn_by_products.csv', index=False)
pd.read_sql_query(query2, conn).to_csv('data/churn_by_country.csv', index=False)
pd.read_sql_query(query3, conn).to_csv('data/churn_by_gender.csv', index=False)
pd.read_sql_query(query4, conn).to_csv('data/churn_by_age.csv', index=False)
#pd.read_sql_query(query5, conn).to_csv('data/churn_by_balance.csv', index=False)

#print("All files exported successfully")

In [4]:
#order balance tier
query5 = """
SELECT
    CASE
        when balance <= 60000 then 'low tier'
        when balance between 60000.01 and 90000 then 'middle tier'
        when balance between 90000.01 and 150000 then 'upper tier'
        else 'high tier'
    end as balance_tier,
    CASE
        when balance <= 60000 then 1
        when balance between 60000.01 and 90000 then 2
        when balance between 90000.01 and 150000 then 3
        else 4
    end as tier_order,
    COUNT(*) as total_customers,
    SUM(churn) as churned_customers,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY balance_tier, tier_order
"""

pd.read_sql_query(query5, conn).to_csv('data/churn_by_balance.csv', index=False)
#print("exported")